In [10]:
import re
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd

# Voor audio cleaning
import librosa
import soundfile as sf
from tqdm.notebook import tqdm


TLT_ROOT    = Path("/Volumes/bigdata3/corpora3/TLT2021/TLT2021/ETLT2021enChallenge")
AUDIO_ROOT  = TLT_ROOT / "audio"
OUTPUT_DIR  = Path.home() / "Downloads" / "ASR_Project_Group_8" / "tlt_english_clean"

SAMPLE_RATE   = 16000     # Whisper-eis
MIN_DURATION  = 1.0       # seconden
MAX_DURATION  = 30.0      # Whisper accepteert max 30s
MIN_WORDS     = 2         # te korte transcripts skippen
TRAIN_RATIO   = 0.80      # 80% sprekers naar train, 20% naar test
RANDOM_SEED   = 42        # voor reproduceerbaarheid

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output gaat naar: {OUTPUT_DIR}")

Output gaat naar: /Users/liekebugel/Downloads/ASR_Project_Group_8/tlt_english_clean


In [11]:

SCRIPTS = [
    AUDIO_ROOT / "NormTrs.pl",
    AUDIO_ROOT / "data_prep.py",
    AUDIO_ROOT / "data_prep2.py",
    TLT_ROOT / "doc" / "TLT2021EvalScript.pl",
]

for script in SCRIPTS:
    if not script.exists():
        print(f"!!! {script.name} niet gevonden\n")
        continue
    print(f"\n{'='*70}")
    print(f"=== {script.name}  ({script.stat().st_size} bytes)")
    print(f"{'='*70}")
    content = script.read_text(encoding="utf-8", errors="replace")
    print(content[:4000])
    if len(content) > 4000:
        print(f"\n[... {len(content)-4000} bytes meer ...]")


=== NormTrs.pl  (885 bytes)
#!/usr/bin/perl

# Roberto Gretter, FBK, 2020

# NormTrs.pl < TLT1618train.orig.trn > TLT1618train.norm.trn

# note: there are numbers written as 123

while($l=<STDIN>){
    $l=~s/\r?\n//; # bastardo dos
    # print "BEFORE <$l>\n";

    $l=~s/^(\S+) / /; # utterance id
    $id=$1;
    $l=" ".$l." ";

    # mapping of known symbols
    $l=~s/<%>/ <unk> /g;
    $l=~s/<non-speech>/ \@ns /g;
    $l=~s/\[sp\]/ /gi; # [SP] means spelling problem

    # removing punctuation
    $l=~s/[\.,;:\?\!\"]/ /g;

    # lowercase
    $l=~s/[A-Z]/\l$&/g;

    # fixing something wrong
    while($l=~s/ \- / /g){}
    $l=~s/\-\-/\-/g;

    # mapping of spontaneous speech phenomena into something easier to handle

    while($l=~s/ uh / \@uh /g){} # in case there are more in sequence
    while($l=~s/ um / \@um /g){} # in case there are more in sequence

    $l=~s/\s+/ /g;
    print "$id$l\n";
}


=== data_prep.py  (1433 bytes)
import os
import csv

# Paths
transcription_file = "/

In [12]:
# Normalisatie die de FBK TLT2021EvalScript.pl ook toepast


EVAL_PATTERNS = [
    # Code-switching parens: @it(io ho gia risposto), @de(...), @en(...), @unk(...)
    (re.compile(r"@(it|de|en|unk)\([^\)]*\)"), " "),
    # <unk> en varianten
    (re.compile(r"<unk(?:-[a-z]+)?>"), " "),
    # Woord-fragmenten (word- of -word)
    (re.compile(r"\s[a-z']+-\s"), " "),
    (re.compile(r"\s-[a-z']+\s"), " "),
    # Losse koppelteken-woorden
    (re.compile(r"\s-\s"), " "),
    # @xxx tokens (@uh, @um, @ns, @voices, etc.)
    (re.compile(r"@[a-z]+"), " "),
    # Resterende speciale tekens (
    (re.compile(r"[\(\#\*\@\)]+"), " "),
    # Multiple spaces collapse
    (re.compile(r"\s+"), " "),
]

def clean_transcript(text: str) -> str:
    """
    Matches TLT2021EvalScript.pl normalization.
  
    """
    text = " " + text.strip().lower() + " "      # padding zoals het Perl script
    for pat, rep in EVAL_PATTERNS:
        text = pat.sub(rep, text)
    return text.strip()


def parse_trn(trn_path: Path) -> dict:
    mapping = {}
    if not trn_path.exists():
        print(f"  ! {trn_path.name} bestaat niet")
        return mapping
    with trn_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if not line.strip():
                continue
 
            if "\t" in line:
                parts = line.split("\t", 1)
            else:
                parts = line.split(" ", 1)
            if len(parts) < 2:
                continue
            audio_id, text = parts[0].strip(), parts[1].strip()
            mapping[audio_id] = text
    return mapping

# Speaker-ID en CEFR extractie blijven hetzelfde
def extract_speaker_id(audio_id: str) -> str:
    m = re.match(r"speaker\w+?_(\d+)-prompt", audio_id)
    if m: return f"spk_{m.group(1)}"
    m = re.match(r"^(\d+)_en_", audio_id)
    if m: return f"spk_{m.group(1)}"
    return f"spk_{audio_id.split('_')[0]}"

def extract_cefr(audio_id: str):
    m = re.search(r"_(A1|A2|B1)_", audio_id)
    return m.group(1) if m else None

In [13]:

test_cases = [
    "i am 10 years old @it(io ho gia' risposto)",
    "my favorite food is @uh pizza and #pasta",
    "hello <unk> goodbye",
    "i live in tren- trento",
    "i like # to play * football",
    "(whispered) hello @ns world",
]

print("Cleaning test:")
for t in test_cases:
    print(f"  IN : {t!r}")
    print(f"  OUT: {clean_transcript(t)!r}\n")

Cleaning test:
  IN : "i am 10 years old @it(io ho gia' risposto)"
  OUT: 'i am 10 years old'

  IN : 'my favorite food is @uh pizza and #pasta'
  OUT: 'my favorite food is pizza and pasta'

  IN : 'hello <unk> goodbye'
  OUT: 'hello goodbye'

  IN : 'i live in tren- trento'
  OUT: 'i live in trento'

  IN : 'i like # to play * football'
  OUT: 'i like to play football'

  IN : '(whispered) hello @ns world'
  OUT: 'whispered hello world'



In [14]:
# Genormaliseerde versie voor TLT1618 (al deels schoongemaakt door FBK)
trans_1618 = parse_trn(AUDIO_ROOT / "TLT1618train.norm.trn")
trans_2017_train = parse_trn(AUDIO_ROOT / "TLT2017train.trn")

print(f"TLT1618 train: {len(trans_1618)} transcripts")
print(f"TLT2017 train: {len(trans_2017_train)} transcripts")

# Voorbeeld
sample_id = next(iter(trans_1618))
print(f"\nVoorbeeld: {sample_id}")
print(f"  ruw:      {trans_1618[sample_id][:120]}")
print(f"  cleaned:  {clean_transcript(trans_1618[sample_id])[:120]}")

TLT1618 train: 11700 transcripts
TLT2017 train: 2299 transcripts

Voorbeeld: speakerIp16_A1_001001001008-promptIp16_A1_en_5_51_100
  ruw:      how old are you
  cleaned:  how old are you


In [15]:
def collect_records(audio_dir: Path, transcripts: dict, source_label: str):
    """Loop door alle .wav files, koppel aan transcript."""
    records = []
    wavs = list(audio_dir.rglob("*.wav"))
    print(f"  {source_label}: {len(wavs)} WAVs gevonden")
    no_match = 0
    for wav in wavs:
        audio_id = wav.stem
        text = transcripts.get(audio_id)
        if text is None:
            no_match += 1
            continue
        records.append({
            "wav_src": wav,
            "audio_id": audio_id,
            "raw_text": text,
            "speaker_id": extract_speaker_id(audio_id),
            "cefr": extract_cefr(audio_id),
            "source": source_label,
        })
    print(f"    -> {len(records)} gekoppeld, {no_match} zonder transcript")
    return records

print("Verzamelen TLT1618train:")
rec_1618 = collect_records(AUDIO_ROOT / "TLT1618train", trans_1618, "TLT1618")
print("\nVerzamelen TLT2017train:")
rec_2017_train = collect_records(AUDIO_ROOT / "TLT2017train", trans_2017_train, "TLT2017")

print(f"\nTotaal trainbaar: {len(rec_1618) + len(rec_2017_train)} samples")

Verzamelen TLT1618train:
  TLT1618: 11700 WAVs gevonden
    -> 11700 gekoppeld, 0 zonder transcript

Verzamelen TLT2017train:
  TLT2017: 2299 WAVs gevonden
    -> 2299 gekoppeld, 0 zonder transcript

Totaal trainbaar: 13999 samples


In [16]:
# Combineer alle 'trainbare' data (TLT1618 + TLT2017train)
trainable = rec_1618 + rec_2017_train
all_speakers = sorted(set(r["speaker_id"] for r in trainable))
print(f"Unieke sprekers: {len(all_speakers)}")

# Random split op spreker-niveau (80% train / 20% test)
random.seed(RANDOM_SEED)
random.shuffle(all_speakers)
n_train = int(len(all_speakers) * TRAIN_RATIO)
train_speakers = set(all_speakers[:n_train])
test_speakers  = set(all_speakers[n_train:])

print(f"Train: {len(train_speakers)} sprekers")
print(f"Test:  {len(test_speakers)} sprekers")

# Wijs elk record een split toe
for r in trainable:
    r["split"] = "train" if r["speaker_id"] in train_speakers else "test"

# Sanity check: speaker overlap?
train_spk = set(r["speaker_id"] for r in trainable if r["split"] == "train")
test_spk  = set(r["speaker_id"] for r in trainable if r["split"] == "test")
print(f"\nOverlap train ∩ test: {len(train_spk & test_spk)}  (moet 0 zijn)")

Unieke sprekers: 3037
Train: 2429 sprekers
Test:  608 sprekers

Overlap train ∩ test: 0  (moet 0 zijn)


In [17]:

import numpy as np

def split_on_silence(y, sr, max_dur, top_db=30):
    """Knip audio in stukjes <= max_dur sec, bij voorkeur op stiltes."""
    max_len = int(max_dur * sr)
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return [(0, len(y))]

    # Groepeer spraak-intervallen tot een stukje vol zit
    chunks = []
    cur_start, cur_end = intervals[0]
    for s, e in intervals[1:]:
        if e - cur_start <= max_len:
            cur_end = e
        else:
            chunks.append((cur_start, cur_end))
            cur_start, cur_end = s, e
    chunks.append((cur_start, cur_end))

    # Eén aaneengesloten spraakstuk > max_len -> hard knippen
    final = []
    for s, e in chunks:
        if e - s <= max_len:
            final.append((s, e))
        else:
            cut = s
            while cut < e:
                final.append((cut, min(cut + max_len, e)))
                cut += max_len
    return final

def distribute_text(text, chunk_durations):
    """Verdeel woorden proportioneel op spraakduur (approximatie)."""
    words = text.split()
    total = sum(chunk_durations) or 1.0
    out, wi = [], 0
    for idx, dur in enumerate(chunk_durations):
        if idx == len(chunk_durations) - 1:
            n = len(words) - wi
        else:
            n = round(len(words) * dur / total)
        out.append(" ".join(words[wi:wi + n]))
        wi += n
    return out

In [21]:
import soundfile as sf
from tqdm.notebook import tqdm

In [22]:
import soundfile as sf

over, total_dur, durs = 0, 0.0, []
for r in tqdm(trainable, desc="Duur scannen"):
    try:
        d = sf.info(str(r["wav_src"])).duration
    except Exception:
        d = librosa.get_duration(path=str(r["wav_src"]))
    durs.append(d)
    total_dur += d
    if d > MAX_DURATION:
        over += 1

n = len(durs)
print(f"Totaal recordings : {n}")
print(f">30s              : {over}  ({100*over/n:.2f}%)")
print(f"Uren in >30s clips: {sum(d for d in durs if d > MAX_DURATION)/3600:.2f} u "
      f"van {total_dur/3600:.2f} u totaal")
print(f"Langste clip      : {max(durs):.1f}s | mediaan: {sorted(durs)[n//2]:.1f}s")

Duur scannen:   0%|          | 0/13999 [00:00<?, ?it/s]

Totaal recordings : 13999
>30s              : 188  (1.34%)
Uren in >30s clips: 5.52 u van 49.17 u totaal
Langste clip      : 180.1s | mediaan: 10.0s


In [25]:
print(f"Totaal te verwerken: {len(trainable)} samples")
processed = []
stats = defaultdict(int)
for r in tqdm(trainable, desc="Audio cleanen (>30s weggooien)"):
    # 1. Transcript clean + filter
    text = clean_transcript(r["raw_text"])
    if not text:
        stats["leeg_transcript"] += 1; continue
    if len(text.split()) < MIN_WORDS:
        stats["te_kort_transcript"] += 1; continue
    # 2. Audio laden, normaliseren, silence trimmen
    try:
        y, sr = librosa.load(str(r["wav_src"]), sr=SAMPLE_RATE, mono=True)
        peak = max(abs(y.max()), abs(y.min()), 1e-9)
        y = y / peak * 0.95
        y, _ = librosa.effects.trim(y, top_db=30)
    except Exception:
        stats["audio_fout"] += 1; continue
    duration = len(y) / SAMPLE_RATE
    if duration < MIN_DURATION:
        stats["te_kort_audio"] += 1; continue
    if duration > MAX_DURATION:          # <-- >30s: weggooien i.p.v. splitsen
        stats["te_lang_weggegooid"] += 1; continue
    # 3. Wegschrijven
    spk_dir = OUTPUT_DIR / r["split"] / r["speaker_id"]
    spk_dir.mkdir(parents=True, exist_ok=True)
    wav_out = spk_dir / (r["audio_id"] + ".wav")
    txt_out = spk_dir / (r["audio_id"] + ".txt")
    sf.write(str(wav_out), y, SAMPLE_RATE, subtype="PCM_16")
    txt_out.write_text(text + "\n", encoding="utf-8")
    processed.append({
        "audio_path": str(wav_out),
        "sentence": text,
        "speaker_id": r["speaker_id"],
        "duration": round(duration, 3),
        "cefr": r["cefr"] or "",
        "source": r["source"],
        "split": r["split"],
        "is_split": False,
    })
    stats["ok_chunks"] += 1
print("\nStatistieken:")
for k, v in stats.items():
    print(f"  {k:25s} {v}")

Totaal te verwerken: 13999 samples


Audio cleanen (>30s weggooien):   0%|          | 0/13999 [00:00<?, ?it/s]


Statistieken:
  ok_chunks                 12937
  leeg_transcript           534
  te_lang_weggegooid        184
  te_kort_transcript        336
  te_kort_audio             8


In [26]:
meta = pd.DataFrame(processed)
meta_path = OUTPUT_DIR / "metadata.csv"
meta.to_csv(meta_path, index=False)
print(f"Metadata opgeslagen: {meta_path}\n")

print("=== Per-split overzicht ===")
for split in ["train", "test"]:
    sub = meta[meta["split"] == split]
    h = sub["duration"].sum() / 3600
    spk = sub["speaker_id"].nunique()
    print(f"  {split:5s}: {len(sub):5d} samples | {h:6.2f} uur | {spk:4d} sprekers")

print(f"\nTotaal: {len(meta)} samples, {meta['duration'].sum()/3600:.2f} uur, "
      f"{meta['speaker_id'].nunique()} sprekers")

print("\n=== Per CEFR-niveau ===")
print(meta.groupby(["split", "cefr"]).size().unstack(fill_value=0))

print("\n=== Per bron ===")
print(meta.groupby(["split", "source"]).size().unstack(fill_value=0))

Metadata opgeslagen: /Users/liekebugel/Downloads/ASR_Project_Group_8/tlt_english_clean/metadata.csv

=== Per-split overzicht ===
  train: 10254 samples |  27.04 uur | 2389 sprekers
  test :  2683 samples |   7.15 uur |  594 sprekers

Totaal: 12937 samples, 34.18 uur, 2983 sprekers

=== Per CEFR-niveau ===
cefr           A1    A2    B1
split                        
test    309  1208   690   476
train  1383  4318  2750  1803

=== Per bron ===
source  TLT1618  TLT2017
split                   
test       2374      309
train      8871     1383


In [27]:
import json
import soundfile as sf

# Map: één manifest (.jsonl) per split. Audio blijft op disk staan als .wav.
HF_DIR = OUTPUT_DIR / "hf_dataset"
HF_DIR.mkdir(parents=True, exist_ok=True)

for split_name in ["train", "test"]:
    sub = meta[meta["split"] == split_name][
        ["audio_path", "sentence", "speaker_id", "duration"]
    ].rename(columns={"audio_path": "audio"})

    manifest_path = HF_DIR / f"{split_name}.jsonl"
    with manifest_path.open("w", encoding="utf-8") as f:
        for rec in sub.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"  {split_name:5s}: {len(sub):5d} samples -> {manifest_path.name}")

print(f"\nManifests opgeslagen in: {HF_DIR}")


def load_split(split_name, sr=SAMPLE_RATE):
    """
    Yield dicts met {'audio': {'array': np.ndarray, 'sampling_rate': sr, 'path': str},
                     'sentence': str, 'speaker_id': str, 'duration': float}.
    Lazy: audio wordt pas geladen wanneer je het record opvraagt.
    """
    manifest = HF_DIR / f"{split_name}.jsonl"
    with manifest.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            audio_arr, file_sr = sf.read(rec["audio"], dtype="float32")
            if file_sr != sr:
                import librosa
                audio_arr = librosa.resample(audio_arr, orig_sr=file_sr, target_sr=sr)
            rec["audio"] = {
                "array": audio_arr,
                "sampling_rate": sr,
                "path": rec["audio"],
            }
            yield rec

# --- Sanity check ---
print("\nVoorbeeld trainsample:")
first = next(load_split("train"))
print(f"  Spreker: {first['speaker_id']}")
print(f"  Duur:    {first['duration']}s")
print(f"  Tekst:   {first['sentence']}")
print(f"  Audio:   shape={first['audio']['array'].shape}, sr={first['audio']['sampling_rate']}")

  train: 10254 samples -> train.jsonl
  test :  2683 samples -> test.jsonl

Manifests opgeslagen in: /Users/liekebugel/Downloads/ASR_Project_Group_8/tlt_english_clean/hf_dataset

Voorbeeld trainsample:
  Spreker: spk_002029001014
  Duur:    10.008s
  Tekst:   yes i like yes i like visit this why it is
  Audio:   shape=(160128,), sr=16000


In [28]:
import sys
!{sys.executable} -m pip show datasets

In [31]:
from IPython.display import Audio, display
import numpy as np

# --- 1. Een willekeurig fragment afspelen ---
sample_meta = meta.sample(16).iloc[0]
audio_arr, sr = sf.read(sample_meta["audio_path"], dtype="float32")

print(f"Spreker:  {sample_meta['speaker_id']}")
print(f"Bron:     {sample_meta['source']} ({sample_meta['cefr'] or 'geen CEFR'})")
print(f"Split:    {sample_meta['split']}")
print(f"Duur:     {sample_meta['duration']}s")
print(f"Tekst:    {sample_meta['sentence']}")
display(Audio(audio_arr, rate=sr))

Spreker:  spk_003009001013
Bron:     TLT1618 (B1)
Split:    train
Duur:     19.704s
Tekst:    i'll prefer i would prefer to live in a town because the town you got many places to go you can go swimming pool or you can go to the gym and instead of many lights you have to you have little shops or


In [32]:
# --- 2. Statistieken: hoe lang zijn de opnames? ---
meta["n_words"] = meta["sentence"].str.split().str.len()

print("=== Duur (seconden) ===")
print(meta.groupby("split")["duration"].describe()[["count","mean","50%","min","max"]].round(2))

print("\n=== Aantal woorden per opname ===")
print(meta.groupby("split")["n_words"].describe()[["count","mean","50%","min","max"]].round(1))

print("\n=== Verdeling woorden (% van train) ===")
bins = [0, 3, 5, 10, 20, 50, 1000]
labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "50+"]
train = meta[meta["split"] == "train"].copy()
train["wbin"] = pd.cut(train["n_words"], bins=bins, labels=labels)
print((train["wbin"].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + " %")

=== Duur (seconden) ===
         count  mean   50%   min    max
split                                  
test    2683.0  9.59  8.64  1.09  23.51
train  10254.0  9.49  8.40  1.02  29.12

=== Aantal woorden per opname ===
         count  mean  50%  min   max
split                               
test    2683.0   9.6  7.0  2.0  54.0
train  10254.0  10.0  6.0  2.0  64.0

=== Verdeling woorden (% van train) ===
wbin
1-3      13.8 %
4-5      27.3 %
6-10     29.0 %
11-20    18.8 %
21-50    11.0 %
50+       0.2 %
Name: proportion, dtype: object
